# 🪙 G-Research Crypto - Starter LGBM Pipeline
![](https://storage.googleapis.com/kaggle-competitions/kaggle/30894/logos/header.png)


### Just a simple pipeline going from zero to a valid submission

We train one `LGBMRegressor` for each asset over a very very naive set of features (the input dataframe `['Count', 'Open', 'High', 'Low', 'Close', 'Volume', 'VWAP']`), we get the predictions correctly using the iterator and we submit. No validation for now, no cross validation... nothing at all lol: just the bare pipeline!


## Please _DO_ upvote if you find this useful!

## References:
* [Detailed API Introduction](https://www.kaggle.com/sohier/detailed-api-introduction)
* [Basic Submission Template](https://www.kaggle.com/sohier/basic-submission-template)
* [Tutorial to the G-Research Crypto Competition](https://www.kaggle.com/cstein06/tutorial-to-the-g-research-crypto-competition)



__Changelog__

__V5__: Added two non-timely features from the tutorial: `Upper_Shadow` and `Lower_Shadow`

This is a fork of the original notebook. My intention here to show is what can be done with this method to improve your score.

# Import and load dfs

References: [Tutorial to the G-Research Crypto Competition](https://www.kaggle.com/cstein06/tutorial-to-the-g-research-crypto-competition)

In [ ]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
import gresearch_crypto


TRAIN_CSV = '/kaggle/input/g-research-crypto-forecasting/train.csv'
ASSET_DETAILS_CSV = '/kaggle/input/g-research-crypto-forecasting/asset_details.csv'

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
asset_df = pd.read_csv(ASSET_DETAILS_CSV)
train_df.head()

In [ ]:
asset_df = asset_df.sort_values('Asset_ID')
asset_df

# Training

## Utility functions to train a model for one asset

In [ ]:
def upper_shadow(df):
    return df['High'] - np.maximum(df['Close'], df["Open"])
def lower_shadow(df):
    return np.minimum(df['Close'], df['Open']) - df['Low']

# A utility function that builds the features from the original dataset
def get_features(df):
    df_feature = df[['Count', 'Open', 'High', 'Low', 'Close', 'Volume', 'VWAP']].copy()
    df_feature['Lower_Shadow'] = lower_shadow(df_feature)
    df_feature['Upper_Shadow'] = upper_shadow(df_feature)
    return df_feature

def get_model(df_train, asset_id):
    df = df_train[df_train['Asset_ID'] == asset_id]
    
    model_df = get_features(df)
    model_df['y'] = df['Target']
    model_df = model_df.dropna(how='any')
    
    X = model_df.drop('y', axis=1)
    y = model_df['y']
    
    model = LGBMRegressor(n_estimators=1000)
    model.fit(X, y)
    return X, y, model

## Loop over all assets

In [ ]:
X_all = {}
y_all = {}
models = {}

for asset_id, asset_name in zip(asset_df['Asset_ID'], asset_df['Asset_Name']):
    print(f"Training model for {asset_name:<16}  (ID={asset_id:<2})")
    X, y, model = get_model(train_df, asset_id)
    X_all[asset_id], y_all[asset_id], models[asset_id] = X, y, model

In [ ]:
#  CHECKING MODEL INTERFACE
x = get_features(train_df.iloc[1])
y_preds = models[0].predict([x])
y_preds[0]

# Predict & submit

References: [Detailed API Introduction](https://www.kaggle.com/sohier/detailed-api-introduction)

Something that helped me understand this iterator was adding a pdb checkpoint inside of the for loop:

```python
import pdb; pdb.set_trace()
```

See [Python Debugging With Pdb](https://realpython.com/python-debugging-pdb/) if you want to use it and you don't know how to.


In [ ]:
env = gresearch_crypto.make_env()
iter_test = env.iter_test()

for i, (df_test, df_pred) in enumerate(iter_test):
    for j , row in df_test.iterrows():
        
        model = models[row['Asset_ID']]
        x_test = get_features(row)
        y_pred = model.predict([x_test])[0]
        
        df_pred.loc[df_pred['row_id'] == row['row_id'], 'Target'] = y_pred
        
        
        # Print just one sample row to get a feeling of what it looks like
        if i == 0 and j == 0:
            display(x_test)
            
    # Display the first prediction dataframe
    if i == 0:
        display(df_pred)

    # Send submissions
    env.predict(df_pred)

In [ ]:
print("Run Faster Damn It!")